# Probing a context, then seeing it in Potree

Walks the whole pipeline on one context:

1. load a Gemma 4 model
2. probe it — for each candidate token, what hidden state would that token take *if it came next*
3. reduce those vectors to 3D with UMAP
4. open the result as an interactive point cloud in the browser

## Before you start

```bash
pip install -e '.[cpu]'          # or '.[gpu]' with an NVIDIA card
./scripts/fetch_potree.sh        # the Potree viewer assets
```

Step 4 also needs the `PotreeConverter` binary at `vendor/potree/PotreeConverter`;
the last cell tells you if it is missing. The matplotlib preview works without it.

## Cost

The model is ~2 GB and downloads once, then lives in the Hugging Face cache.
Probing is the slow part, so this notebook probes a **few thousand tokens**
rather than the full 262k vocabulary — enough to see the geometry, small
enough to finish on a laptop. `experiments/extract_context_embeddings.py`
does the full sweep.

## Running without model weights

Set `SMOKE = True` below to swap in a small randomly-initialised model and a
toy tokenizer. Every later cell is unchanged, so the notebook runs end to end
in seconds — useful for checking the plumbing. The numbers mean nothing.

In [ ]:
import os

# The context to probe. The model sees this, and we ask what every
# candidate token would look like as the next one.
CONTEXT = "There is a mole in the middle of"

# How many tokens to probe. A few thousand is plenty to see structure.
N_PROBE = 4000

# True -> tiny random model, no download. See the note above.
SMOKE = os.environ.get("LIFT_NOTEBOOK_SMOKE") == "1"

print(f"context: {CONTEXT!r}")
print(f"probing {N_PROBE} tokens" + ("  [SMOKE MODE]" if SMOKE else ""))

## 1. Load the model

`build_model()` pulls the weights from Hugging Face the first time and reuses
the local cache after that — you do not re-download on later runs. The
safetensors-to-JAX conversion does happen every time you build, which is why
a notebook is a good place for this: the model stays in memory across cells.

In [ ]:
import time

if SMOKE:
    # A small randomly-initialised model, built through the same code path
    # as the real one. gemma4_12b takes its shapes from a config dict, so no
    # monkeypatching is needed.
    import jax, numpy as np
    from lift.models.gemma4_12b import Gemma4Config, from_weights

    cfg = Gemma4Config.from_dict(dict(
        vocab_size=512, hidden_size=64, intermediate_size=128, num_hidden_layers=6,
        num_attention_heads=4, num_key_value_heads=2, head_dim=16, global_head_dim=32,
        sliding_window=64, max_position_embeddings=256,
    ))
    key = jax.random.PRNGKey(0)
    def rnd(shape, k):
        return np.asarray(jax.random.normal(jax.random.PRNGKey(k), shape), np.float32) * 0.05
    raw = {"embed_tokens.weight": rnd((cfg.vocab_size, cfg.hidden_size), 1),
           "norm.weight": np.ones(cfg.hidden_size, np.float32)}
    for i in range(cfg.num_hidden_layers):
        hd, kv = cfg.layer_head_dim(i), cfg.layer_kv_heads(i)
        inter = cfg.layer_intermediate(i)
        raw |= {
            f"layers.{i}.self_attn.q_proj.weight": rnd((cfg.num_attention_heads * hd, cfg.hidden_size), 10 + i),
            f"layers.{i}.self_attn.k_proj.weight": rnd((kv * hd, cfg.hidden_size), 20 + i),
            f"layers.{i}.self_attn.v_proj.weight": rnd((kv * hd, cfg.hidden_size), 30 + i),
            f"layers.{i}.self_attn.o_proj.weight": rnd((cfg.hidden_size, cfg.num_attention_heads * hd), 40 + i),
            f"layers.{i}.self_attn.q_norm.weight": np.ones(hd, np.float32),
            f"layers.{i}.self_attn.k_norm.weight": np.ones(hd, np.float32),
            f"layers.{i}.mlp.gate_proj.weight": rnd((inter, cfg.hidden_size), 50 + i),
            f"layers.{i}.mlp.up_proj.weight": rnd((inter, cfg.hidden_size), 60 + i),
            f"layers.{i}.mlp.down_proj.weight": rnd((cfg.hidden_size, inter), 70 + i),
            f"layers.{i}.input_layernorm.weight": np.ones(cfg.hidden_size, np.float32),
            f"layers.{i}.post_attention_layernorm.weight": np.ones(cfg.hidden_size, np.float32),
            f"layers.{i}.pre_feedforward_layernorm.weight": np.ones(cfg.hidden_size, np.float32),
            f"layers.{i}.post_feedforward_layernorm.weight": np.ones(cfg.hidden_size, np.float32),
            f"layers.{i}.layer_scalar": np.ones(1, np.float32),
        }
    model = from_weights(cfg, raw)
    from lift.models import gemma4_12b as modelling

    class ToyTokenizer:
        """Character-level stand-in so the notebook runs without weights."""
        bos_token_id = 2
        def __call__(self, text, add_special_tokens=False):
            return {"input_ids": [3 + (ord(c) % 500) for c in text.split(" ") and text][:64]}
        def decode(self, ids):
            return "".join(chr(65 + (int(i) % 26)) for i in ids)
    tokenizer = ToyTokenizer()
else:
    from lift.models import gemma4 as modelling
    t0 = time.time()
    model, tokenizer = modelling.build_model()
    print(f"built in {time.time() - t0:.0f}s")

print("vocab x hidden:", model.embed.shape)

## 2. The context, as the model reads it

Every probe will sit at the position right after this run of tokens.

In [ ]:
import jax.numpy as jnp

context_ids = jnp.array(
    [tokenizer.bos_token_id] + tokenizer(CONTEXT, add_special_tokens=False)["input_ids"]
)
n = int(context_ids.shape[0])

for i, tid in enumerate(context_ids):
    print(f"  {i:>2}  {int(tid):>7}  {tokenizer.decode([int(tid)])!r}")
print(f"\n{n} tokens -> probes will sit at position {n}")

## 3. Pick the tokens to probe

A random slice of the vocabulary is mostly noise. This takes the model's own
top predictions for this context (the tokens it actually considers), adds the
four words the study tracks, and pads with a spread of the vocabulary so the
cloud has a background to sit against.

In [ ]:
import jax
import numpy as np

logits = model(context_ids)
probs = jax.nn.softmax(logits[-1])

top_probs, top_ids = jax.lax.top_k(probs, k=min(50, int(probs.shape[0])))
print("the model's top continuations:")
for tid, p in list(zip(top_ids, top_probs))[:10]:
    print(f"  {tokenizer.decode([int(tid)])!r:<16} {float(p):.3f}")

vocab = int(model.embed.shape[0])
wanted = [int(t) for t in top_ids]

if not SMOKE:
    from lift.study import WORDS_OF_INTEREST
    wanted += [tid for _, tid in WORDS_OF_INTEREST]
    print("\ntracking the four senses:", [w for w, _ in WORDS_OF_INTEREST])

# pad out with an even spread of the vocabulary
rng = np.random.default_rng(0)
background = rng.choice(vocab, size=min(N_PROBE, vocab), replace=False)
probe_ids = list(dict.fromkeys(wanted + [int(i) for i in background]))[:N_PROBE]

# remember where the interesting ones ended up, to mark them later
highlight = list(range(len(dict.fromkeys(wanted))))
print(f"\nprobing {len(probe_ids)} tokens; {len(highlight)} of them marked")

## 4. Probe

One masked forward pass per chunk. Each probe token attends to the whole
context and to itself, never to the other probes, and every one of them is
pinned to RoPE position `n` — they are all standing in for the same slot.

The result is one vector per probed token: that token's representation *in
this context*.

In [ ]:
t0 = time.time()
E = modelling.get_contextual_embeddings(
    model, context_ids,
    token_ids=probe_ids,
    chunk_size=256,
    save=False,
)
print(f"\n{E.shape} in {time.time() - t0:.1f}s")

## 5. Check the probe is doing what we think

A probe row should equal the last hidden state of an ordinary causal forward
pass on `[context..., token]`. If the RoPE positions were wrong, only the
first token of each chunk would match — so this is worth confirming before
reading anything into the geometry.

In [ ]:
check_slot = len(probe_ids) // 2
token = probe_ids[check_slot]

probe_logits = E[check_slot] @ np.asarray(model.embed).T
true_logits = np.asarray(model(jnp.concatenate([context_ids, jnp.array([token])]))[-1])

cap = getattr(modelling, "LOGIT_SOFTCAP", None) or getattr(model.config, "final_logit_softcapping", None)
if cap:
    probe_logits = np.tanh(probe_logits / cap) * cap

err = np.abs(probe_logits - true_logits).max() / np.abs(true_logits).max()
print(f"token {token} at chunk slot {check_slot}")
print(f"relative difference from a true causal forward: {err:.2e}")
print("OK" if err < 1e-4 else "MISMATCH - do not trust what follows")

## 6. Reduce to 3D

UMAP with a cosine metric. `backend="auto"` uses cuML on an NVIDIA card and
falls back to umap-learn on the CPU — the CPU path is fine at this size and
slow on a full vocabulary.

In [ ]:
from lift.geometry.dimred import reduce, backend_in_use

print("backend:", backend_in_use("auto"))
t0 = time.time()
cloud = reduce(E, n_neighbors=15, min_dist=0.1, random_state=0)
print(f"{E.shape} -> {cloud.shape} in {time.time() - t0:.1f}s")

## 7. A quick look before the real viewer

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from lift.viz.plots import plot_3d_umap

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(projection="3d")
plot_3d_umap(ax, cloud, CONTEXT, highlight=np.array(highlight),
             labels=np.array([tokenizer.decode([probe_ids[i]]) for i in highlight]),
             n_labels=min(6, len(highlight)))
plt.tight_layout()
plt.show()

## 8. Save it, and open it in Potree

Potree streams the cloud with level-of-detail, so it stays smooth at sizes
matplotlib cannot handle — the point of using it is that you can do the full
262k vocabulary this way. Clicking a point asks the server to decode that
index back into a token.

In [ ]:
from lift.paths import DATA_DIR, POTREE_BUILD, POTREE_CONVERTER

out = DATA_DIR / "reduced" / "notebook_cloud.npy"
out.parent.mkdir(parents=True, exist_ok=True)
np.save(out, np.asarray(cloud, dtype=np.float32))
print(f"saved {cloud.shape} -> {out}")

# The probed token ids, so a click can be mapped back to a token.
np.save(out.with_name("notebook_cloud_token_ids.npy"), np.array(probe_ids, np.int32))

print()
if not POTREE_BUILD.exists():
    print("Potree viewer assets missing -> run ./scripts/fetch_potree.sh")
elif not POTREE_CONVERTER.exists():
    print(f"PotreeConverter missing at {POTREE_CONVERTER}")
    print("Build it from source; see the header of scripts/fetch_potree.sh.")
else:
    print("ready. Run this in a terminal:")
print(f"\n    lift-potree {out}\n")
print("then open http://localhost:8080")

### Launching it from here instead

`lift-potree` runs a blocking web server, so it is better started in a
terminal. If you would rather launch it from the notebook, run the cell
below and interrupt the kernel to stop it.

In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "lift.viz.potree", str(out)])

---

## Where to go next

- `experiments/extract_context_embeddings.py` runs the full vocabulary for
  every prefix of a context, which is what the study actually analyses.
- `experiments/reduce_embeddings.py` reduces those in bulk.
- `experiments/morph_gif.py` animates the move between two reduced clouds.

The full sweep writes a matrix per prefix, each `vocab x hidden` — roughly
1.5 GB for Gemma 4 E2B. Check your disk before starting one.